In [ ]:
# ===========================================================================
# UA-SPEECH DATA PIPELINE - interactive driver
#
# All logic lives in the src/ package; this notebook only calls it, so the
# notebook and run_pipeline.py cannot drift apart. To change behaviour, edit
# the module, not this notebook.
#
#   src/config.py         paths, speaker ground truth, label maps, hyperparams
#   src/extraction.py     .tgz archive extraction
#   src/scanning.py       filename parsing, verification, mic filter, labels
#   src/visualization.py  EDA figures
#   src/splits.py         LOSO (detection) and balanced 81-fold (severity)
#   src/preprocessing.py  resample, VAD trim, pad, MFCC
#   src/dataset.py        UASpeechDataset
#   src/models/           deep / acoustic / fusion pathways
# ===========================================================================

# STAGE 0 - Setup. Put the project root on the import path; autoreload picks up
# edits to src/ without a kernel restart.
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import config
from src.console import print_header, print_kv

config.ensure_directories()

print_header("UA-Speech Dysarthria Pipeline")
print_kv("Project root", config.PROJECT_ROOT)
print_kv("Archive folder", config.ARCHIVE_DIR)
print_kv("Audio folder", config.AUDIO_DIR)
print_kv("Ground truth", f"{len(config.CONTROL_IDS)} controls + "
                        f"{len(config.DYSARTHRIC_IDS)} dysarthric = "
                        f"{len(config.ALL_SPEAKERS)} speakers")
print_kv("Microphone channel", config.TARGET_MIC)
print_kv("Words per speaker", config.WORDS_PER_SPEAKER)

In [ ]:
# STAGE 1 - Extract the archives.
# Copy UASpeech_normalized_C.tgz and UASpeech_normalized_FM.tgz into
# data/archives/ first. Runs once; the extracted audio persists in
# data/extracted/, so skip this cell on later passes.
from src.extraction import extract_all_archives

extract_all_archives()

In [ ]:
# STAGE 2 - Scan and verify against the 28-speaker ground truth.
# Filenames parse as <Speaker>_<Block>_<WordCode>_<Mic>.wav. Two rules baked
# into parse_filename, both of which the original exploratory scan got wrong:
#   1. The mic channel is taken POSITIONALLY from the final token. The old
#      logic searched for the first token starting with 'M', which matched male
#      speaker IDs like M01 before ever reaching the real mic token.
#   2. macOS resource-fork duplicates ('._' prefix) are skipped - the archive
#      holds one per real .wav, which doubled the apparent file count.
from src.scanning import scan_audio_files, verify_speakers

df_audio = scan_audio_files()
speakers_ok = verify_speakers(df_audio)

df_audio.head()

In [ ]:
# STAGE 3 - EDA. Figures render inline and are also saved to outputs/figures/
# at 300 dpi, ready for the IEEE paper draft.
from src.visualization import run_eda

run_eda(df_audio, show=True)

In [ ]:
# STAGE 4 - Filter to microphone channel M6 (base-paper protocol: M6 only, all
# three blocks, all word categories, no word-type filtering).
# Expect 21,420 utterances: 11,475 dysarthric + 9,945 healthy control.
from src.scanning import filter_mic_channel

df_m6 = filter_mic_channel(df_audio)

In [ ]:
# STAGE 5 - Per-speaker word counts; every speaker should have all 765 words on
# M6. Speakers below that are flagged, NOT dropped: an incomplete speaker is
# still usable, and silently removing one would change the LOSO fold count.
from src.scanning import check_word_counts

word_counts = check_word_counts(df_m6)

In [ ]:
# STAGE 6 - Severity labels: four classes (Very Low / Low / Mid / High) for the
# 15 dysarthric speakers; controls get 'N/A (Control)'.
from src.scanning import add_severity_labels

df_m6 = add_severity_labels(df_m6)

df_m6.head()

In [ ]:
# STAGE 7 - Cross-validation splits.
#   Detection: Leave-One-Speaker-Out over all 28 speakers -> 28 folds.
#   Severity:  classes are unbalanced (4/3/3/5 speakers), so
#              config.DROPPED_FOR_BALANCE excludes M12, M08, F05 to reach 3 per
#              class -> 3^4 = 81 leave-one-per-class-out iterations.
#
# NOTE: the base paper gives no explicit exclusion list, so that set of three
# speakers is OUR ASSUMPTION. Confirm with the team before treating any
# severity result as final.
from src.splits import build_severity_folds, get_loso_split, summarize_detection_splits

summarize_detection_splits(df_m6)
severity_folds = build_severity_folds(df_m6)

In [ ]:
# STAGE 8 - Build the PyTorch dataset. UASpeechDataset returns the raw waveform
# (Deep Pathway), the 39-dim MFCC tensor (Acoustic Pathway), both labels, and
# the speaker ID - so both pathways train on identical audio and splits.
from src.dataset import UASpeechDataset

dataset = UASpeechDataset(df_m6)
sample = dataset[0]

print_header("Dataset Build")
print_kv("Total samples", len(dataset))
print_kv("Waveform shape", tuple(sample["waveform"].shape))
print_kv("MFCC shape", tuple(sample["mfcc"].shape))
print_kv("Detection label", sample["group_label"].item())
print_kv("Severity label", sample["severity_label"].item())
print_kv("Speaker ID", sample["speaker_id"])

manifest_path = config.OUTPUT_DIR / "m6_manifest.csv"
df_m6.to_csv(manifest_path, index=False)
print_kv("Manifest saved", manifest_path)

In [ ]:
# STAGE 9 - Fusion model shape check. One forward pass to confirm the pathways
# line up: Deep 768 + Acoustic 128 -> classifier consumes 896.
# Downloads the wav2vec 2.0 weights on first run.
# Architecture check only - training is the Fusion Architect's task, not built.
import torch
from torch.utils.data import DataLoader

from src.models.fusion import FusionModel

loader = DataLoader(dataset, batch_size=2, shuffle=True)
batch = next(iter(loader))

model = FusionModel(num_classes=2)  # 2 = detection task; use 4 for severity
model.eval()

with torch.no_grad():
    logits = model(batch["waveform"].squeeze(1), batch["mfcc"])

print_header("Fusion Model Shape Check")
print_kv("Waveform batch", tuple(batch["waveform"].squeeze(1).shape))
print_kv("MFCC batch", tuple(batch["mfcc"].shape))
print_kv("Output logits", tuple(logits.shape))
print_kv("LoRA parameters", model.deep_pathway.trainable_parameter_summary())